# Baseline DSA via `lic_dsf.dsa`

Build Ext + Macro books, then the full **Output 1-1 / 1-2** tables
(index = Excel row, columns = years — same geometry as the spreadsheet).

See `docs/07-baseline-dsa.qmd`.

In [1]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

from lic_dsf.dsa import (
    BaselineExternalBook,
    BaselinePublicBook,
)
from lic_dsf.load import (
    load_external_debt_inputs,
    load_instruments_from_workbook,
    load_lc_nr_instruments_from_workbook,
    load_macro_debt_inputs,
)
from lic_dsf.output import output_11_table, output_12_table
from lic_dsf.pv import (
    ExternalDebtBook,
    MacroDebtBook,
    PVPortfolio,
)

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "demo":
    REPO_ROOT = REPO_ROOT.parent

WORKBOOK = REPO_ROOT / "data" / "lic-dsf-template-2025-08-12.xlsx"

pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 80)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

WORKBOOK

PosixPath('/home/sravan/py-lic-dsf/data/lic-dsf-template-2025-08-12.xlsx')

In [2]:
instruments = load_instruments_from_workbook(
    WORKBOOK, include_zero_disbursement=True
)
lc_nr = load_lc_nr_instruments_from_workbook(
    WORKBOOK, include_zero_disbursement=True
)
ext = ExternalDebtBook(
    portfolio=PVPortfolio(instruments=tuple(instruments) + tuple(lc_nr)),
    inputs=load_external_debt_inputs(WORKBOOK),
)
macro = MacroDebtBook(inputs=load_macro_debt_inputs(WORKBOOK), external=ext)
ext_base = BaselineExternalBook(macro=macro, external=ext)
pub_base = BaselinePublicBook(macro=macro, external=ext)

# Display window: last hist year through first decade of projections (as on Output 1-x).
first = macro.inputs.first_projection_year
years = list(range(first - 2, first + 11))
ext_base.years[0], ext_base.years[-1], first, years[0], years[-1]

(2011, 2044, 2024)

## Output 1-1 — External DSA (Excel row index)

In [3]:
out_11 = output_11_table(ext_base)
out_11.loc[:, [y for y in years if y in out_11.columns]]

,2022,2023,2024,2025,2026
PV of PPG external debt / GDP,NaN,42.9240,44.8846,43.1514,41.2265
PV of PPG external debt / exports,NaN,105.1065,109.0647,103.4689,99.5562
PV of PPG external debt / revenue,NaN,259.0434,247.7666,237.3607,221.1705
PPG debt service / exports,12.3440,14.7305,17.1681,16.5074,15.1705
PPG debt service / revenue,31.3280,36.3045,39.0015,37.8686,33.7022
External GFN (USD),"1,488.5526","1,337.0348","2,360.2673","2,091.5041","2,101.2626"


## Output 1-2 — Public DSA (Excel row index)

In [ ]:
out_12 = output_12_table(pub_base)
out_12.loc[:, [y for y in years if y in out_12.columns]]